<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/4_Aprendizaje_no_supervisado/2_Taller_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller: Análisis de Patrones de Consumo Internacional con Apriori**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**

- Andres Santiago Cristiano Trujillo
- Samuel David Huertas Infante
- Juan Diego Villabón López

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma:“Taller_Apriori_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/qERdEpXpmx.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

21 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

**Caso de Estudio: Consultoría para Global Retail Inc.**

**Contexto:** Una firma multinacional de e-commerce, "Global Retail Inc.", te ha contratado como consultor de datos. La empresa opera en múltiples países y ha notado que sus ventas y la efectividad de sus campañas de marketing varían significativamente entre regiones. Su hipótesis es que los patrones de compra y las asociaciones de productos son diferentes en cada mercado.

**Tu Misión:** Analizar el historial de transacciones de la empresa para descubrir y comparar las reglas de asociación de productos para dos de sus mercados más importantes en Latinoamérica: México y Colombia. Tu objetivo final es entregar recomendaciones de negocio accionables (ej. estrategias de cross-selling, promociones personalizadas) basadas en los patrones de consumo que descubras en cada país.

**Dataset:** Encuentra mayor información en el archivo "diccionario_alimentos_retail_top30.xlsx".

## Ejercicio 1: Configuración Inicial, Carga y Exploración de Datos

1.1 Importa las librerías necesarias

In [7]:
%pip install mlxtend
import os
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

In [8]:
# Configuraciones de visualización
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

In [9]:
from google.colab import drive, files
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1.2 Carga el dataset "alimentos_retail_top30.csv" que se encuentra en el repositorio del curso, carpeta "datasets". El dataframe debe llamarse "df".

In [10]:
df = pd.read_csv('/content/drive/MyDrive/2026-i-curso-ia-para-economia/datasets/alimentos_retail_top30.csv', header=0)

In [11]:
# Debe ser (6899, 8)
print("Dimensiones del DataFrame:")
print(df.shape)

Dimensiones del DataFrame:
(6899, 8)


In [12]:
print("\nInformación general del DataFrame:")
df.info()


Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6899 entries, 0 to 6898
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    6899 non-null   object 
 1   StockCode    6899 non-null   int64  
 2   Description  6899 non-null   object 
 3   Quantity     6899 non-null   int64  
 4   InvoiceDate  6899 non-null   object 
 5   UnitPrice    6899 non-null   float64
 6   CustomerID   6879 non-null   float64
 7   Country      6899 non-null   object 
dtypes: float64(2), int64(2), object(4)
memory usage: 431.3+ KB


1.3 Revisa si hay valores nulos en alguna columna y cuántos son

In [13]:
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,20
Country,0


1.4 Genera las estadísticas descriptivas de las variables numéricas

In [14]:
df.describe()

,StockCode,Quantity,UnitPrice,CustomerID
count,"6,899.00","6,899.00","6,899.00","6,879.00"
mean,"55,544.94",3.00,3.42,"15,024.12"
std,"25,875.73",1.43,1.06,"1,732.95"
min,"26,907.00",-5.00,1.65,"12,000.00"
25%,"31,048.00",2.00,2.36,"13,524.00"
50%,"42,889.00",3.00,3.39,"15,041.00"
75%,"87,297.00",4.00,4.44,"16,530.50"
max,"95,931.00",5.00,4.90,"17,999.00"


1.5 Observando las salidas del ejercicio anterior, ¿qué problemas potenciales identificas en las columnas CustomerID y Quantity?

## Ejercicio 2: Limpieza y Preprocesamiento de Datos

Los datos del mundo real rara vez son perfectos. Antes de cualquier análisis, debemos "sanear" nuestro dataset. Completa el código en cada paso según las instrucciones.

Crea un nuevo dataframe llamado "df_limpio" para los siguientes puntos.

2.1 **Manejo de Valores Nulos**: Las transacciones sin un CustomerID no son útiles para nosotros, ya que no podemos agrupar las compras de un cliente específico.

In [15]:
# TAREA: Elimina todas las filas donde 'CustomerID' es nulo.
### TU CÓDIGO AQUÍ ###
df_limpio = df.dropna(subset=['CustomerID'])

In [16]:
# El tipo de dato de CustomerID debe ser entero
### TU CÓDIGO AQUÍ ###
df_limpio['CustomerID'] = df_limpio['CustomerID'].astype(int)

2.2 **Limpieza de Descripciones de Productos** Las descripciones pueden tener espacios en blanco al inicio o al final que podrían hacer que un mismo producto se cuente como dos diferentes.

In [17]:
# TAREA: # Verifica cuántas descripciones únicas hay.
### TU CÓDIGO AQUÍ ###
df['Description'].nunique()


25

In [18]:
# TAREA: Limpia la columna 'Description' eliminando espacios extra al inicio y al final.
### TU CÓDIGO AQUÍ ###
df['Description'] = df['Description'].str.strip()


In [19]:
# TAREA: Verifica cuántas descripciones únicas quedaron después de la limpieza.
### TU CÓDIGO AQUÍ ###
df['Description'].nunique()

20

2.3 **Filtrado de Transacciones Anómalas**: Las facturas (InvoiceNo) que empiezan con 'C' indican una cancelación. Estas no son compras reales y deben ser eliminadas. Del mismo modo, las cantidades (Quantity) negativas representan devoluciones.

In [20]:
# TAREA: Elimina las filas que correspondan a cancelaciones.
### TU CÓDIGO AQUÍ ###
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

In [21]:
# TAREA: Elimina las filas con cantidades negativas.
### TU CÓDIGO AQUÍ ###
df = df[df['Quantity'] >= 0]


In [22]:
# Verifiquemos las dimensiones del DataFrame después de la limpieza. Debe ser (6864, 8)
df_limpio.shape

(6879, 8)

## Ejercicio 3: Análisis Comparativo por País

Ahora que los datos están limpios, vamos a segmentarlos y a aplicar el algoritmo Apriori para encontrar los patrones de compra en México y Colombia.

**Preparación de la Cesta de Mercado (Función)**

La siguiente función toma un dataframe, lo agrupa por factura y descripción, y lo transforma en el formato de matriz binaria que necesita el algoritmo Apriori. Estudia esta función, no necesitas modificarla.

In [23]:
def preparar_cesta(dataframe, pais):
    """Filtra por país y prepara la matriz de transacciones."""

    # Filtrar por el país de interés
    df_pais = dataframe[dataframe['Country'] == pais]

    # Crear la cesta: agrupar productos por factura
    cesta = (df_pais.groupby(['InvoiceNo', 'Description'])['Quantity']
             .sum().unstack().reset_index().fillna(0)
             .set_index('InvoiceNo'))

    # Convertir todas las cantidades positivas a 1 y todo lo demás a 0
    cesta_encoded = (cesta > 0).astype(int)

    return cesta_encoded

3.1 Análisis para México

In [24]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de México. Almacena el resultado en la variable cesta_mx.
### TU CÓDIGO AQUÍ ###
cesta_mx = preparar_cesta(df, "México")

In [25]:
# TAREA: Aplica el algoritmo apriori para encontrar itemsets con un soporte mínimo de 2%.
# Almacena el resultado en la variable frequent_itemsets_mx.
# Muestra los 10 itemsets con el soporte más alto.
### TU CÓDIGO AQUÍ ###
frequent_itemsets_mx = apriori(cesta_mx, min_support=0.02, use_colnames=True)

In [26]:
# TAREA: Genera las reglas de asociación. Queremos reglas con un Lift mayor a 2. Almacena el resultado en la variable rules_mx.
### TU CÓDIGO AQUÍ ###
rules_mx = rules = association_rules(frequent_itemsets_mx, metric="lift", min_threshold=2)

In [27]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
### TU CÓDIGO AQUÍ ###
rules = rules.sort_values(by=['lift', 'confidence'], ascending=[False, False])
rules[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']].head(10)


,antecedents,consequents,antecedent support,consequent support,confidence,lift
88,"(CILANTRO, TOMATE)","(CEBOLLA, CHILE JALAPEÑO)",0.32,0.32,0.94,2.90
93,"(CEBOLLA, CHILE JALAPEÑO)","(CILANTRO, TOMATE)",0.32,0.32,0.92,2.90
89,"(CILANTRO, CEBOLLA)","(TOMATE, CHILE JALAPEÑO)",0.31,0.33,0.96,2.88
92,"(TOMATE, CHILE JALAPEÑO)","(CILANTRO, CEBOLLA)",0.33,0.31,0.90,2.88
90,"(CILANTRO, CHILE JALAPEÑO)","(TOMATE, CEBOLLA)",0.33,0.33,0.92,2.80
91,"(TOMATE, CEBOLLA)","(CILANTRO, CHILE JALAPEÑO)",0.33,0.33,0.91,2.80
10,"(LIMÓN, AGUACATE)",(TOTOPOS),0.27,0.33,0.93,2.79
15,(TOTOPOS),"(LIMÓN, AGUACATE)",0.33,0.27,0.74,2.79
62,"(AGUACATE, QUESO FRESCO, TOTOPOS)",(LIMÓN),0.03,0.35,0.96,2.72
69,(LIMÓN),"(AGUACATE, QUESO FRESCO, TOTOPOS)",0.35,0.03,0.07,2.72


3.3 Observa las 3 reglas con el Lift más alto para México (1, 3 y 5). **Interprétalas:** ¿Qué te dicen estas asociaciones? ¿Qué tipo de productos son?

Los productos del lift mas alto indican que si un consumidor compra cilantro y tomate, muy probablemente tambien comprara cebolla chile y jalapeños, tambien sucede al contrario.

Este mismo comportamiento se ve en el segundo y el tercer lift mas alto, sin embargo cambiando de productos, para el segundo siendo si compra cilantro y cebolla, comprara tomate chile y jalapeño, y para el tercero, si compra tomate y cebollar comprara cilantro chile y jalapeño.

Aqui podemos ver los productos principales del mercado los cuales son aquellos que siempre estan presentes en todas las canastas de estos 3 lifts, siendo estos:

- Cebolla
- Chile
- Cilantro
- Jalapeño
- Tomate

3.4 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

El soporte del antecedente y del consecuente para el primer lift es de 0.32, lo que significa que el 32% de las compras tienen productos del antecedente y 32% tendran el consecuente si se cumple la relación, lo cual es muy probable ya que tenemos una confianza del 0.94, lo que indica que en el 94% de los casos cada que ocurra el antecedente ocurrira el consecuente. Por último el valor del lift nos deja ver que la probabilidad conjunta es 2.9 veces mayor a si fueran independientes, asi que hay una dependencia positiva fuerte.

Para el segundo y el tercer lift la interpretación sera escencialmente la misma, sin embargo para el segundo lift; cuando la canasta antecedente es de cebolla y cilantro, hay mas confianza de que se cumpla el consecuente de una forma considerable (6%).

3.5 **Recomendación de Negocio:** Basado en estas reglas, ¿qué promoción o estrategia de venta específica podrías sugerir para el mercado mexicano?

Como los productos estan fuertemente vinculados mexico podrifomentar ventas conjuntas mediante paquetes, promociones cruzadas o ubicaciones estrategicas en tienda, con el objetivo de aumentar el valor de la compra aprovechando la complementariedad de los productos.

3.6 Análisis para Colombia

In [28]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de Colombia. Almacena el resultado en la variable cesta_co.
### TU CÓDIGO AQUÍ ###
cesta_co = preparar_cesta(df, "Colombia")

In [29]:
# TAREA: Aplica el algoritmo apriori con un soporte mínimo del 2%.
# Almacena el resultado en la variable frequent_itemsets_co.
# Muestra los 10 itemsets con el soporte más alto.
### TU CÓDIGO AQUÍ ###
frequent_itemsets_co = apriori(cesta_co, min_support=0.02, use_colnames=True)


In [30]:
# TAREA: Genera las reglas de asociación con un Lift mayor a 2. Almacena el resultado en la variable rules_co.
### TU CÓDIGO AQUÍ ###
rules_co = association_rules(frequent_itemsets_co, metric="lift", min_threshold=2)

In [31]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
### TU CÓDIGO AQUÍ ###
rules_co = rules_co.sort_values(by=['lift', 'confidence'], ascending=[False, False])
rules_co[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']].head(10)


,antecedents,consequents,antecedent support,consequent support,confidence,lift
40,"(FRIJOL CARGAMANTO, AZÚCAR, CAFÉ)",(LECHE),0.05,0.39,0.96,2.43
41,(LECHE),"(FRIJOL CARGAMANTO, AZÚCAR, CAFÉ)",0.39,0.05,0.11,2.43
12,"(AZÚCAR, CAFÉ)",(LECHE),0.32,0.39,0.95,2.42
13,(LECHE),"(AZÚCAR, CAFÉ)",0.39,0.32,0.76,2.42
4,"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",(ARROZ),0.30,0.38,0.92,2.41
9,(ARROZ),"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",0.38,0.30,0.73,2.41
55,"(PAN TAJADO, AZÚCAR, CAFÉ)",(LECHE),0.03,0.39,0.94,2.38
62,(LECHE),"(PAN TAJADO, AZÚCAR, CAFÉ)",0.39,0.03,0.07,2.38
46,"(HUEVOS, AZÚCAR, CAFÉ)",(LECHE),0.04,0.39,0.92,2.35
51,(LECHE),"(HUEVOS, AZÚCAR, CAFÉ)",0.39,0.04,0.09,2.35


3.7 Observa las 3 reglas con el Lift más alto para Colombia (1, 3 y 5). **Interprétalas:** ¿Qué patrones de consumo específicos del mercado colombiano revelan estas reglas? ¿Son diferentes a las de México?

En colombia se puede ver que hay mas consumo de productos básicos como café, azúcar, leche, arroz y frijol, donde el soporte del consecuente (leche o arroz) es relativamente alto (0.38–0.39), indicando que son bienes ampliamente consumidos, mientras que los antecedentes tienen soportes entre 0.03 y 0.32. La confianza varía entre 0.92 y 0.96 en algunas reglas, lo que refleja que estos productos suelen comprarse en conjunto dentro de patrones de consumo diario más que en preparaciones específicas.

3.8 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

En el primer lift podemos ver que cuando el antecedente es de frijol, azúcar y café, hay una gigantesca probabilidad de comprar leche, sin embargo cuando el antecedente es de leche, la probabilidad baja significativamente de un 96% a un 7%, lo que indica que la leche es un prodcuto dependiente de otros. Sin embargo es muy probable que la persona compre café y azúcar si compra leche, esto debido a la complementariedad de estos productos en diversas preparaciones.

3.9 **Recomendación de Negocio:** ¿Qué campaña de marketing (diferente a la de México) podrías diseñar para los clientes colombianos?

Dado que productos como la leche aparecen en cerca del 39% de las compras y que la probabilidad condicional de compra conjunta puede superar el 90%, con lifts superiores a 2.3, la estrategia más adecuada es incentivar compras frecuentes mediante promociones por volumen y combos de productos básicos, enfocándose en aumentar la recurrencia y fidelización del consumidor más que en ventas cruzadas específicas.